# One-off: rebuild silver PARENTS as latest-version-per-id

**Step 2 of the approved silver rebuild plan (Nov 9 2026).** Single full-history pass over bronze, dedupe to the **latest version per PK**, atomic full overwrite in silver — so children can then be rebuilt cleanly against one-row-per-id parents.

Key decisions (user-approved contract):

* **Full history**: `parse_bronze_table(watermark=None, silver_name=None)` — read ALL retained bronze (since Aug 24 2026), no CDC window, no payload anti-join; byte-identical re-dumps already collapse via `payload_hash`.
* **One row per id**: `row_number() = 1` ordered by `bronze_insert_date DESC, bronze_updated_at DESC` within the family PK — the newest payload version wins.
* **Atomic overwrite**: `write_silver` full-load mode (`_CDC_WATERMARK = None` → `mode("overwrite")` + `overwriteSchema=true`), same table names, same write path the Nov 4 rebuild proved.
* **PK pins**: calendly `EVENT_URI` (find_pk could not assign one), sentiment `CHANNEL_ID`, mdl `ID`, and activities `id` (pinned so find_pk's composite id+lead_id can never split a version chain).
* **Nightly schema parity**: per-family args exactly as the 2_* task notebooks (all use `flatten=False`); `extract_custom(dry_run=True)` builds the `custom_fields` MAP on the parent.

Safety:

* `RUN_REBUILD = False` by default — every family cell is a **dry run**: parse + dedupe + counts, **no writes**. Flip to `True` only after the dry-run numbers are reviewed and execution is explicitly approved.
* Each family is its own cell, run in order (leads → activities → crm users → custom → calendly → sentiment → mdl) — a failure stops only that family.
* Run **between** nightlies (not in the ~13:00–13:20 UTC window) — children are NOT touched by this notebook; do not run child notebooks until all 7 parents are rebuilt.
* After the rebuild, `get_silver_watermark()` = `MAX(bronze_updated_at)` of the new table = newest bronze load, so nightly CDC resumes with no-new-data and skips versions silver already holds via the `payload_hash` anti-join.

Live run COMPLETED Nov 9 2026 (~18:50 UTC, user-authorized): leads 380,681 → 64,615; activities 390,899 → 1,013,155; crm users 166,528 → 444 (443 ids + 1 NULL-id source record); custom 1,873 → 30 (29 ids + 1 NULL-id record); calendly 105,193 → 102,071; sentiment 149,774 → 2,258; mdl 29,699 → 26,261. Verified on disk: rows == distinct ids on all 7 (NULL-id records excepted); CDC watermarks preserved at the Nov 9 13:01–13:02 loads.


In [0]:
%run ../utils


In [0]:
# ============================================================================
# REBUILD CONFIG — edit flags here, not in the engine
# ============================================================================

# Master switch. False = dry-run (parse + dedupe + report counts, NO writes).
# Flip to True ONLY after the dry-run numbers are reviewed and the user has
# explicitly approved the overwrite.
RUN_REBUILD = False

# Per-family engine args, exactly as the nightly 2_* task notebooks use them.
# (bronze_name, silver_name, flatten, pk)
#   flatten: as passed to process_table (all 7 nightlies use False).
#   pk: the dedupe key. Pinned per family where find_pk is unreliable (see md cell).
FAMILIES = [
    ('leads_raw',                  'leads_processed',                    False, 'id'),
    ('lead_activites_raw',         'lead_activites_processed',           False, 'id'),
    ('close_crm_users_raw',        'close_crm_users_processed',          False, 'id'),
    ('custom_activites_raw',       'custom_activites_processed',         False, 'id'),
    ('calendly_scheduled_events',  'calendly_scheduled_events_processed', False, 'EVENT_URI'),
    ('student_sentiment',          'student_sentiment_processed',        False, 'CHANNEL_ID'),
    ('mdl_users_raw',              'mdl_users_processed',                False, 'ID'),
]

assert all(isinstance(f[2], bool) for f in FAMILIES), "flatten must be explicit bool per family"
assert all(f[3] for f in FAMILIES), "every family needs a pinned PK"
print(f"Config: {len(FAMILIES)} families | RUN_REBUILD = {RUN_REBUILD} "
      f"({'LIVE OVERWRITE' if RUN_REBUILD else 'DRY RUN — no writes'})")


In [0]:
# ============================================================================
# REBUILD ENGINE — full-history parse, latest-version-per-id, atomic overwrite
# ============================================================================
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
import time


def rebuild_parent(bronze_name, silver_name, flatten=False, pk=None, run=False):
    """One-off parent rebuild: full bronze history -> latest version per PK -> atomic overwrite.

    Mirrors process_table's full-load path (parse_bronze_table with NO watermark
    reads ALL of bronze, byte-identical re-dumps collapse via payload_hash),
    then dedupes to ONE row per PK before the write, so silver holds the current
    state, not the version log.

    PK pinning: a pinned pk always wins; for lead_activites we pin 'id' because
    find_pk can pick a composite (id+lead_id) — every child joins on id alone,
    so the dedupe MUST run within id, not within (id, lead_id).

    Steps — identical to the nightly full path so the written schema matches:
      1. parse_bronze_table(bronze_name)      # full history, no watermark
      2. flatten_structs(skip_custom=True)    # only if flatten=True (never for our 7)
      3. pinned / find_pk
      4. extract_custom(dry_run=True)         # custom_fields MAP on parent only
      5. dedupe: row_number over Window(partitionBy(pk).orderBy(
             bronze_insert_date DESC NULLS LAST, bronze_updated_at DESC NULLS LAST,
             payload_hash DESC)) == 1
      6. run=False: report counts only; run=True: write_silver full overwrite
    """
    global _CDC_WATERMARK
    _CDC_WATERMARK = None  # full-load mode: write_silver overwrites, no delete+append

    t0 = time.time()
    print(f"\n{'='*80}")
    print(f"🔧 {bronze_name} → {silver_name}  (PK: {pk or 'find_pk'})")
    print(f"{'='*80}")

    # -- 1-2. Full-history parse --------------------------------------------
    df_flat = parse_bronze_table(bronze_name, watermark=None, silver_name=None)
    if df_flat is None:
        print("  ⛔ ABORT: parse returned no data — silver untouched.")
        return None
    if flatten:
        df_flat = flatten_structs(df_flat, skip_custom=True)

    # -- 3. PK: pinned value wins ---------------------------------------------
    if pk and pk not in df_flat.columns:
        print(f"  ⛔ ABORT: pinned PK '{pk}' not in parsed columns — silver untouched.")
        return None
    dedupe_pk = pk or find_pk(df_flat, table_name=silver_name)
    if not dedupe_pk:
        print("  ⛔ ABORT: no PK could be assigned — silver untouched.")
        return None
    print(f"  🗝️ Dedupe PK: {dedupe_pk}")

    # -- 4. Custom fields -> custom_fields MAP on parent -----------------------
    carry = list(dict.fromkeys(["bronze_insert_date", "bronze_updated_at", dedupe_pk]))
    has_custom = "custom" in df_flat.columns and isinstance(df_flat.schema["custom"].dataType, StructType)
    has_custom_cf = any(c.startswith("custom_cf_") for c in df_flat.columns)
    if has_custom or has_custom_cf:
        pipeline = PIPELINE_NUMBERS.get(silver_name, 1)
        df_flat = extract_custom(df_flat, silver_name, carry, f"{pipeline}_1",
                                 parent_pk=dedupe_pk, dry_run=True)

    # -- 5. Latest-version-per-id dedupe --------------------------------------
    n_versions = df_flat.count()
    n_null_upd = df_flat.filter(col("bronze_updated_at").isNull()).count()
    if n_null_upd:
        print(f"  ⚠️ {n_null_upd:,} version rows have NULL bronze_updated_at (legacy stamping gap)")
    # nulls-last: an unstamped legacy row must never outrank a stamped newer
    # load; payload_hash as final key makes the pick fully deterministic.
    w = Window.partitionBy(dedupe_pk).orderBy(
        col("bronze_insert_date").desc_nulls_last(),
        col("bronze_updated_at").desc_nulls_last(),
        col("payload_hash").desc())
    df_latest = (df_flat.withColumn("_rn", row_number().over(w))
                 .filter("_rn = 1").drop("_rn"))
    n_latest = df_latest.count()
    n_ids = df_latest.select(dedupe_pk).distinct().count()

    # Version distribution (how many ids carry how many versions).
    dist = (df_flat.groupBy(dedupe_pk).count()
            .withColumnRenamed("count", "_v")
            .groupBy("_v").count()
            .withColumnRenamed("count", "_n_ids")
            .orderBy("_v")).collect()
    dist_str = ", ".join(f"{r['_n_ids']:,} ids × {r['_v']}v" for r in dist[:6])
    if len(dist) > 6:
        dist_str += f", … {len(dist) - 6} more version tiers"

    # Existing silver row count for the before/after report.
    try:
        before = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{silver_name}").count()
    except Exception:
        before = None

    print(f"  📊 Full history: {n_versions:,} version rows → {n_latest:,} latest rows "
          f"({n_ids:,} distinct {dedupe_pk})")
    print(f"  📈 Versions per id: {dist_str}")
    if before is not None:
        print(f"  📊 Silver before: {before:,} rows")

    # -- 6. Write (only when explicitly authorised) ----------------------------
    if not run:
        print(f"  🧪 DRY RUN — no write. (RUN_REBUILD=True overwrites {silver_name})")
        for temp in _TEMP_TABLES:
            spark.sql(f"DROP TABLE IF EXISTS {temp}")
        _TEMP_TABLES.clear()
        return {"silver": silver_name, "versions": n_versions, "latest": n_latest,
                "ids": n_ids, "before": before, "written": False}

    write_silver(df_latest, silver_name)
    for temp in _TEMP_TABLES:
        spark.sql(f"DROP TABLE IF EXISTS {temp}")
    _TEMP_TABLES.clear()

    after = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{silver_name}").count()
    before_str = f"{before:,} → " if before is not None else ""
    print(f"  ✅ {silver_name}: {before_str}{after:,} rows (latest per id)")

    return {"silver": silver_name, "versions": n_versions, "latest": n_latest,
            "ids": n_ids, "before": before, "after": after, "written": True}


In [0]:
# ============================================================================
# LEADS family: bronze leads_raw, PK id. Sweep expectation: 64,615 ids.
# Children (contacts, contacts_emails, custom) rebuild in the child step.
# ============================================================================
results = []
bronze_name, silver_name, flatten, pk = FAMILIES[0]
r = rebuild_parent(bronze_name, silver_name, flatten=flatten, pk=pk, run=RUN_REBUILD)
if r:
    results.append(r)


In [0]:
# ============================================================================
# ACTIVITIES family: bronze lead_activites_raw, PK id (PINNED — find_pk prefers
# composite id+lead_id, but every child joins on id alone: _to, email_body,
# attendees, envelope all carry activity_id → parent id).
# Sweep expectation: ~1M distinct ids in bronze (silver holds 211K today).
# Children NOT written here — child-rebuild step handles them.
# ============================================================================
bronze_name, silver_name, flatten, pk = FAMILIES[1]
r = rebuild_parent(bronze_name, silver_name, flatten=flatten, pk=pk, run=RUN_REBUILD)
if r:
    results.append(r)


In [0]:
# ============================================================================
# CLOSE CRM USERS: bronze close_crm_users_raw, PK id.
# Sweep: ×375 versions — 166,528 silver rows for ~444 distinct ids.
# ============================================================================
bronze_name, silver_name, flatten, pk = FAMILIES[2]
r = rebuild_parent(bronze_name, silver_name, flatten=flatten, pk=pk, run=RUN_REBUILD)
if r:
    results.append(r)


In [0]:
# ============================================================================
# CUSTOM ACTIVITIES: bronze custom_activites_raw, PK id.
# Sweep: ×62 versions — 1,873 silver rows, 30 ids / 163 bronze rows.
# Children (editable_with_roles, fields, fields_editable_with_roles) rebuild
# in the child step, not here.
# ============================================================================
bronze_name, silver_name, flatten, pk = FAMILIES[3]
r = rebuild_parent(bronze_name, silver_name, flatten=flatten, pk=pk, run=RUN_REBUILD)
if r:
    results.append(r)


In [0]:
# ============================================================================
# CALENDLY: bronze calendly_scheduled_events, PK EVENT_URI (explicit — find_pk
# could NOT assign a PK on a 200-row sample). Sweep: ~101K distinct EVENT_URIs
# in bronze; silver today holds every version (~105K rows).
# ============================================================================
bronze_name, silver_name, flatten, pk = FAMILIES[4]
r = rebuild_parent(bronze_name, silver_name, flatten=flatten, pk=pk, run=RUN_REBUILD)
if r:
    results.append(r)


In [0]:
# ============================================================================
# STUDENT SENTIMENT: bronze student_sentiment, PK CHANNEL_ID (find_pk agrees).
# Sweep: ×66 versions — 149,774 silver rows for 2,258 channel ids.
# ============================================================================
bronze_name, silver_name, flatten, pk = FAMILIES[5]
r = rebuild_parent(bronze_name, silver_name, flatten=flatten, pk=pk, run=RUN_REBUILD)
if r:
    results.append(r)


In [0]:
# ============================================================================
# MDL USERS: bronze mdl_users_raw, PK ID (find_pk agrees).
# Sweep: ×1.1 versions — 29,699 silver rows for 26,261 ids.
# ============================================================================
bronze_name, silver_name, flatten, pk = FAMILIES[6]
r = rebuild_parent(bronze_name, silver_name, flatten=flatten, pk=pk, run=RUN_REBUILD)
if r:
    results.append(r)


In [0]:
# ============================================================================
# SUMMARY — one line per family; verify against sweep expectations before any
# child rebuild is attempted.
# ============================================================================
print(f"\n{'='*80}")
print(f"📋 PARENT REBUILD SUMMARY  ({'LIVE' if RUN_REBUILD else 'DRY RUN'})")
print(f"{'='*80}")
for r in results:
    status = "✅ written" if r.get("written") else "🧪 dry-run"
    b = f"{r['before']:,} → " if r.get("before") is not None else ""
    a = f"{r['after']:,}" if r.get("after") is not None else f"{r['latest']:,} (planned)"
    print(f"  {status}  {r['silver']}: {b}{a} rows  ({r['ids']:,} ids from {r['versions']:,} versions)")

if not RUN_REBUILD:
    print("\n  Next: review counts vs sweep expectations; if correct, set RUN_REBUILD=True, re-run all, then rebuild children.")
else:
    print("\n  Next: rebuild child tables (activity family _to/email_body/attendees/envelope + leads-family children).")


In [0]:
# ============================================================================
# MERGE SMOKE TEST — validates the SILVER_MERGE_KEYS / _merge_silver CDC path
# in utils cell 6 (Nov 9 2026), landed right after this rebuild. Re-run any
# time after a utils change; self-cleaning (creates, verifies and drops a
# throwaway silver table _merge_smoke_test, and removes its registry entry).
# Requires the %run ../utils cell to have been re-run AFTER the utils edit.
# ============================================================================
from datetime import datetime

TEST = "_merge_smoke_test"
FULL = f"{CATALOG}.{SILVER_SCHEMA}.{TEST}"

# 0. Fresh throwaway table + registry entry
spark.sql(f"DROP TABLE IF EXISTS {FULL}")
spark.sql(f"""
CREATE TABLE {FULL} (
  id STRING, name STRING,
  bronze_insert_date TIMESTAMP, bronze_updated_at TIMESTAMP, payload_hash STRING
)
""")
SILVER_MERGE_KEYS[TEST] = "id"

_ts = lambda s: datetime.strptime(s, "%Y-%m-%d %H:%M:%S")

# 1. Batch 1: two brand-new ids -> INSERT x2
_CDC_WATERMARK = _ts("2026-11-09 00:00:00")
df1 = spark.createDataFrame(
    [("a1", "alpha", _ts("2026-11-08 10:00:00"), _ts("2026-11-09 12:00:00"), "h1"),
     ("a2", "beta",  _ts("2026-11-08 10:00:00"), _ts("2026-11-09 12:00:00"), "h2")],
    "id string, name string, bronze_insert_date timestamp, bronze_updated_at timestamp, payload_hash string")
write_silver(df1, TEST)
n1 = spark.table(FULL).count()
assert n1 == 2, f"batch1: expected 2 rows, got {n1}"
print(f"batch 1 OK: {n1} rows (new ids inserted)")

# 2. Batch 2: a1 CHANGED twice in one batch (dedupe to newest), a3 NEW,
#    one NULL-PK row, plus a schema-evolution column 'extra'
df2 = spark.createDataFrame(
    [("a1", "alpha-stale", _ts("2026-11-08 10:00:00"), _ts("2026-11-09 12:00:00"), "h1",  None),
     ("a1", "alpha-v2",    _ts("2026-11-08 11:00:00"), _ts("2026-11-09 13:00:00"), "h1b", "x"),
     ("a3", "gamma",       _ts("2026-11-08 11:00:00"), _ts("2026-11-09 13:00:00"), "h3",  "y"),
     (None, "null-pk",     _ts("2026-11-08 11:00:00"), _ts("2026-11-09 13:00:00"), "h4",  None)],
    "id string, name string, bronze_insert_date timestamp, bronze_updated_at timestamp, payload_hash string, extra string")
write_silver(df2, TEST)

_rows = spark.table(FULL)
n2 = _rows.count()
a1_name = _rows.filter("id = 'a1'").first()["name"]
has_extra = "extra" in _rows.columns
null_pk_rows = _rows.filter("id IS NULL").count()
print(f"batch 2: {n2} rows | a1.name = {a1_name} | 'extra' column present = {has_extra} | NULL-PK rows = {null_pk_rows}")

assert n2 == 4, f"expected 4 rows (a1 kept single + a3 + null-pk), got {n2}"
assert a1_name == "alpha-v2", f"a1 should UPDATE in place to alpha-v2, got {a1_name}"
assert has_extra, "schema evolution failed: 'extra' column not added"
assert null_pk_rows == 1, f"NULL-PK row missing: {null_pk_rows}"

print("\nALL MERGE SMOKE TESTS PASSED: INSERT, in-place UPDATE, batch dedupe, NULL-PK passthrough, schema evolution")

# 3. Cleanup
del SILVER_MERGE_KEYS[TEST]
_CDC_WATERMARK = None
spark.sql(f"DROP TABLE IF EXISTS {FULL}")
print(f"cleanup done: {FULL} dropped, registry entry removed")
